# Fehleranalyse des Endmodells

**Autor:** jd · **Datum:** 2026-10-30

**Ziel:** Kennzahlen und Fehler des Endmodells `combined_svm_aez_month` (Run `17dc9e16`) auf dem 80/20-Holdout für den Abschlussbericht (Kapitel 6). Die Analyse in `01_jd_fehleranalyse` stammt aus der 70/30-Zeit und nutzt ältere Modelle; hier werden ihre Befunde für das Endmodell geprüft.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support

from awp2.config import (
    AEZ_COL,
    CROP_COL,
    MODEL_DOCS_FIGURES_DIR,
    MONTH_COL,
    RANDOM_FOREST_N_ESTIMATORS,
    SEED,
    STAGE_COL,
)
from awp2.data import load_split
from awp2.data.split import combined_label
from awp2.evaluation import compare_predictions, error_breakdown, plot_confusion_matrices
from awp2.experiment import RunConfig, out_of_fold_predictions, run
from awp2.models import make_combined_svm, make_hierarchical_svm, oracle_crop_metrics
from awp2.plots import plot_error_breakdown, plot_error_rate_by, save_doc_figure
from awp2.preprocessing import PreprocessingConfig

In [ ]:
split = load_split()
X_val, y_val = split.X_val, split.y_val
X_val.shape

## 1. Endmodell und Vergleichsmodelle

Das Endmodell mit den eingefrorenen Einstellungen aus `tools/predict_combined_svm_aez_month.py`, dazu die Random-Forest-Baseline und die hierarchische SVM (Einstellungen aus `hier_reduction_all_bands`, `514156e7`), alle auf dem 80/20-Trainingsteil neu trainiert (`track=False`, keine neuen Läufe). Die Validierung dient nur der Analyse, es wird nichts eingestellt.

In [ ]:
models = {
    "Baseline (RF)": (
        "baseline",
        RandomForestClassifier(
            n_estimators=RANDOM_FOREST_N_ESTIMATORS,
            max_depth=20,
            max_features=0.3,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1,
        ),
        PreprocessingConfig(use_cyclic_month=False),
    ),
    "Endmodell (SVM)": (
        "combined",
        make_combined_svm().set_params(estimator__C=300.0, estimator__gamma=0.003),
        PreprocessingConfig(use_spectral_standard_scale=True, use_aez_month=True),
    ),
    "Hierarchisch (SVM)": (
        "hierarchical",
        make_hierarchical_svm().set_params(
            crop_estimator__C=30.0,
            crop_estimator__gamma=0.01,
            stage_estimator__C=100.0,
            stage_estimator__gamma=0.003,
        ),
        PreprocessingConfig(use_spectral_standard_scale=True),
    ),
}
results = {
    label: run(
        model,
        RunConfig(
            name="final_error_analysis_refit",
            approach=approach,
            preprocessing=preprocessing,
            track=False,
            log_model=False,
        ),
    )
    for label, (approach, model, preprocessing) in models.items()
}
predictions = {label: result.y_pred for label, result in results.items()}
final = predictions["Endmodell (SVM)"]
pd.DataFrame({label: result.metrics.model_dump() for label, result in results.items()}).round(3)

## 2. Wo entstehen die Fehler?

Jede Zeile ist ganz richtig, hat die falsche Kultur oder nur das falsche Stadium. Die Oracle-Kultur zeigt für die hierarchische SVM, was ohne Kulturfehler möglich wäre.

In [ ]:
breakdowns = {label: error_breakdown(y_val, pred) for label, pred in predictions.items()}
breakdown_ax = plot_error_breakdown(breakdowns)
breakdown_ax.set_title("Fehlerzerlegung auf dem Holdout (80/20)")
save_doc_figure(breakdown_ax.figure, "endmodell_zerlegung", MODEL_DOCS_FIGURES_DIR)
pd.DataFrame({label: b.model_dump() for label, b in breakdowns.items()}).round(3)

In [ ]:
oracle = oracle_crop_metrics(results["Hierarchisch (SVM)"].pipeline, X_val, y_val)
round(oracle.bacc_combined, 3), round(oracle.bacc_stage, 3)

## 3. Confusion Matrices des Endmodells

Zeilennormiert: Anteil je wahrer Klasse.

In [ ]:
confusion_figure = plot_confusion_matrices(y_val, final)
save_doc_figure(confusion_figure, "endmodell_konfusion", MODEL_DOCS_FIGURES_DIR)

## 4. Precision, Recall und Fallzahl je Klasse

Für Kultur, Stadium und die 23 kombinierten Klassen; schwächster Recall zuerst.

In [ ]:
def per_class(true: pd.Series, pred: pd.Series) -> pd.DataFrame:
    labels = sorted(true.unique())
    precision, recall, f1, support = precision_recall_fscore_support(
        true, pred, labels=labels, zero_division=0
    )
    return pd.DataFrame(
        {"Precision": precision, "Recall": recall, "F1": f1, "Fälle": support}, index=labels
    ).sort_values("Recall")


per_class(y_val[CROP_COL], final[CROP_COL]).round(3)

In [ ]:
per_class(y_val[STAGE_COL], final[STAGE_COL]).round(3)

In [ ]:
combined_table = per_class(combined_label(y_val), combined_label(final))
combined_table.round(3)

In [ ]:
small = combined_table["Fälle"] < 20
pd.DataFrame(
    {
        "Klassen": [small.sum(), (~small).sum()],
        "mittlerer Recall": [
            combined_table.loc[small, "Recall"].mean(),
            combined_table.loc[~small, "Recall"].mean(),
        ],
        "mittlere Precision": [
            combined_table.loc[small, "Precision"].mean(),
            combined_table.loc[~small, "Precision"].mean(),
        ],
    },
    index=["< 20 Fälle", "≥ 20 Fälle"],
).round(3)

## 5. Häufigste Verwechslungen des Endmodells

In [ ]:
crop_wrong = {label: pred[CROP_COL] != y_val[CROP_COL] for label, pred in predictions.items()}
pd.DataFrame({"wahr": y_val[CROP_COL], "vorhergesagt": final[CROP_COL]}).loc[
    crop_wrong["Endmodell (SVM)"]
].value_counts().head(6)

In [ ]:
true_pair, predicted_pair = combined_label(y_val), combined_label(final)
stage_only = ~crop_wrong["Endmodell (SVM)"] & (true_pair != predicted_pair)
pd.DataFrame({"wahr": true_pair, "vorhergesagt": predicted_pair}).loc[
    stage_only
].value_counts().head(6)

## 6. Wann und wo liegt die Kultur falsch?

Kulturfehlerrate nach Monat und AEZ. Gegenprobe ohne Holdout: dieselben Raten aus Out-of-fold-Vorhersagen des Endmodells auf dem Trainingsteil.

In [ ]:
context_figure, context_axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, (column, label) in zip(context_axes, [(MONTH_COL, "Monat"), (AEZ_COL, "AEZ")], strict=True):
    plot_error_rate_by(crop_wrong, X_val[column], xlabel=label, ax=ax)
    ax.set_title(f"Kulturfehler nach {label}")
context_figure.tight_layout()
save_doc_figure(context_figure, "endmodell_kulturfehler_kontext", MODEL_DOCS_FIGURES_DIR)

In [ ]:
final_model, final_preprocessing = models["Endmodell (SVM)"][1:]
oof = out_of_fold_predictions(
    final_model,
    RunConfig(
        name="final_error_analysis_oof",
        approach="combined",
        preprocessing=final_preprocessing,
        track=False,
    ),
)
oof_crop_wrong = oof[CROP_COL] != split.y_train[CROP_COL]
final_crop_wrong = crop_wrong["Endmodell (SVM)"]
(
    pd.DataFrame(
        {
            "Holdout": final_crop_wrong.groupby(X_val[MONTH_COL]).mean(),
            "Training (out-of-fold)": oof_crop_wrong.groupby(split.X_train[MONTH_COL]).mean(),
            "Zeilen Holdout": X_val[MONTH_COL].value_counts(),
        }
    ).round(3),
    pd.DataFrame(
        {
            "Holdout": final_crop_wrong.groupby(X_val[AEZ_COL]).mean(),
            "Training (out-of-fold)": oof_crop_wrong.groupby(split.X_train[AEZ_COL]).mean(),
            "Zeilen Holdout": X_val[AEZ_COL].value_counts(),
        }
    ).round(3),
)

## 7. Unterscheiden sich die Modelle wirklich?

Exakter McNemar-Test auf kombinierte Treffer (nicht BAcc, siehe `compare_predictions`).

In [ ]:
pd.DataFrame(
    {
        f"Endmodell vs. {other}": compare_predictions(y_val, final, predictions[other]).model_dump()
        for other in ["Baseline (RF)", "Hierarchisch (SVM)"]
    }
).T

## Fazit

- Das Endmodell reproduziert Run `17dc9e16`: BAcc Kultur 0,923, Stadium 0,922, kombiniert 0,869; +0,118 gegenüber der neu trainierten Baseline (0,751).
- 89,1 % der Zeilen sind ganz richtig; rund drei Viertel der Fehler sind Kulturfehler (8,0 % gegenüber 2,9 % nur Stadium). Häufigste Verwechslungen: Soja → Mais (22), Winterweizen → Baumwolle (17), Mais → Soja (12).
- Schwächste Klassen sind frühe Stadien und Seneszenz (`soybean|Emerge_VEarly` 0,48, `cotton|Mature_Senesc` 0,62). Seltene Klassen (< 20 Fälle) sind im Mittel nicht schlechter (0,91 gegenüber 0,85), aber unsicher.
- Kulturfehler häufen sich im Mai, Juni und Oktober sowie in AEZ 9, 7 und 5 – auch out-of-fold auf dem Trainingsteil; die Befunde aus `01_jd_fehleranalyse` gelten damit auch für das Endmodell.
- Hierarchisch kosten Kulturfehler rund 0,10 (Oracle-Kultur 0,941 statt 0,845).